# M2–M5 — đối chứng lựa chọn 30 tuyến

**M là mã phương pháp, không phải trọng số hoặc số thứ tự bắt buộc.** Có thể chạy M2/M3 độc lập sau khi đã có cùng pool 60 và dữ liệu kiểm tra; M4/M5 được lấy lại từ lịch sử chọn tuyến đã lưu.

Chọn **Python thường** có NumPy và pandas. Notebook chỉ gọi module, không khởi động QGIS, không gửi HTTP và không thay bộ 30 gốc. [Hướng dẫn đầy đủ](../docs/lez/SELECTION_BENCHMARKS.md).


In [8]:
from pathlib import Path
import importlib
import importlib.util
import json
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/lez/selection_benchmarks.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở thư mục work trong VS Code rồi chạy notebook từ work/notebooks.")
missing = [name for name in ("numpy", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Kernel thiếu: " + ", ".join(missing)
                       + ". Chọn Python có thư viện trong requirements-notebook.txt.")
scripts = str(ROOT / "scripts")
if scripts not in sys.path:
    sys.path.insert(0, scripts)
bench = importlib.import_module("lez.selection_benchmarks")
if Path(bench.__file__).resolve() != (ROOT / "scripts/lez/selection_benchmarks.py").resolve():
    raise RuntimeError("Kernel đang giữ module từ dự án khác. Restart Kernel rồi chạy lại.")
import pandas as pd
from IPython.display import display
print("Đã tìm thấy work; đang dùng Python thường:", sys.executable)


Đã tìm thấy work; đang dùng Python thường: /Users/quanh.hg/python/.pyqgis-runtime/bin/python3.12


### 1. Kiểm tra dữ liệu trước khi chạy

Kiểm đầu vào, hash, ma trận 12 ô, quota và lịch sử bộ 30. Thiếu bundle hoặc kiểm tra không đạt thì dừng; không dựng lại GIS hoặc tự tải dữ liệu để lấp chỗ thiếu. Sau khi clone, cần khôi phục dữ liệu như [hướng dẫn máy mới](../docs/setup/GETTING_STARTED.md).


In [9]:
check = bench.preflight_benchmarks(ROOT)
print(json.dumps(check, ensure_ascii=False, indent=2))


{
  "status": "ready",
  "fingerprint": "489317219bd2c59527ba36722e64d29933f22c9519e91cae5f189107ad516bac",
  "methods": [
    "M2",
    "M3",
    "M4",
    "M5"
  ],
  "runtime": {
    "python": "3.12.11",
    "numpy": "2.5.2"
  },
  "network_calls": 0,
  "qgis_started": false,
  "candidate_pool": {
    "A": 20,
    "B": 20,
    "C": 20
  },
  "selected_quota": {
    "A": 10,
    "B": 8,
    "C": 12
  },
  "seed_count": 10,
  "M0_run_required": false,
  "M1_status": "allocation_rule_not_finalized",
  "baseline_modified": false
}


### 2. Chọn các phương pháp cần chạy

- **M2:** mỗi lượt chọn tuyến hợp lệ tạo D thấp nhất; 10 cặp khởi đầu cố định, không hoán đổi.
- **M3:** mỗi lượt chọn tuyến bổ sung nhiều mét đường RC1–RC4 chưa được phủ nhất; cùng 10 khởi đầu xếp theo D để kiểm soát cách bắt đầu, không hoán đổi. Sau khởi đầu, M3 không dùng D để chọn tuyến hoặc chọn lượt chạy tốt nhất.
- **M4:** bộ 30 trước hoán đổi, đọc từ lịch sử chọn C1–C7 đã kiểm chứng.
- **M5:** bộ 30 sau hoán đổi trong cùng lịch sử; chính là bộ gốc được dùng đối chiếu.

Có thể chọn chỉ `["M2"]` hoặc `["M3"]`; thứ tự trong danh sách không quyết định kết quả. **M1 còn chờ chốt quy tắc phân bổ; không đưa M1 vào danh sách.** M0 đã có notebook riêng, không bốc lại ở đây.


In [10]:
METHODS = ["M2", "M3", "M4", "M5"]


### 3. Chạy hoặc tiếp tục

Các phương pháp dùng cùng quota **A10/B8/C12**, điều kiện tiền khảo sát và trùng hành lang **<70%**. Không ép có sẵn 11/12 ô; số ô là kết quả đánh giá. Ctrl+C giữ kết quả từng phương pháp đã hoàn thành; chạy lại cell để dùng lại cache hợp lệ. Phương pháp chưa hoàn thành có thể phải chạy lại các khởi đầu từ đầu.


In [11]:
report = bench.run_benchmarks(ROOT, methods=METHODS, progress=print)
print("Trạng thái:", report.get("status"))
print({name: result.get("status") for name, result in report["methods"].items()})


M2: dùng lại kết quả đã kiểm tra hash và chỉ tiêu
M3: dùng lại kết quả đã kiểm tra hash và chỉ tiêu
M4: dùng lại kết quả đã kiểm tra hash và chỉ tiêu
M5: dùng lại kết quả đã kiểm tra hash và chỉ tiêu
Trạng thái: completed_pre_survey_benchmarks
{'M2': 'completed', 'M3': 'completed', 'M4': 'completed', 'M5': 'completed'}


### 4. So sánh bằng cùng mẫu số

D dùng chiều dài theo lượt đi RC1–RC4; độ phủ dùng chiều dài hợp không trùng. Connector chỉ nằm trong tổng km hành trình. D thấp hơn và độ phủ cao hơn là hai mục tiêu khác nhau; xem cả km, trùng lặp và số ô. C6 chưa có bằng chứng vẫn chưa được xác minh.


In [12]:
rows = []
for method, result in report["methods"].items():
    row = {"method": method, "status": result.get("status")}
    row.update(result.get("metrics", {}))
    rows.append(row)
comparison = pd.json_normalize(rows)
columns = ["method", "status", "D_combined", "D_by_domain.inside", "D_by_domain.outside",
           "traversed_length_m", "unique_rc_length_m", "unique_rc_coverage_fraction",
           "max_unique_corridor_overlap_fraction", "material_cells", "quantile_groups_covered"]
display(comparison[[c for c in columns if c in comparison.columns]])


,method,status,D_combined,D_by_domain.inside,D_by_domain.outside,traversed_length_m,unique_rc_length_m,unique_rc_coverage_fraction,max_unique_corridor_overlap_fraction,material_cells,quantile_groups_covered
0,M2,completed,0.377814,0.532860,0.310093,266927.830347,236092.404157,0.010528,0.596699,10,33
1,M3,completed,0.485520,0.579032,0.377596,338357.546349,295693.277874,0.013185,0.496621,11,33
2,M4,completed,0.368532,0.426114,0.310093,274352.009252,241557.839524,0.010771,0.471389,11,33
3,M5,completed,0.365187,0.427502,0.310093,274031.310637,240898.855865,0.010742,0.596699,11,33


### 5. Xem 30 mã tuyến và các khởi đầu

Bộ được chọn là kết quả tốt nhất tìm thấy trong các khởi đầu hoàn thành, chưa chứng minh tối ưu toàn cục. Nếu một khởi đầu bế tắc hoặc một phương pháp thất bại, giữ lý do trong kết quả để kiểm tra, không tự nới điều kiện.


In [13]:
selected = [{"method": method, "status": result.get("status"),
             "selected_route_ids": ", ".join(result.get("selected_route_ids", [])),
             "source": result.get("provenance", {}).get("source"),
             "history_path": result.get("history_path")}
            for method, result in report["methods"].items()]
display(pd.DataFrame(selected))

seed_rows = []
for method, result in report["methods"].items():
    for seed_result in result.get("seed_runs", []):
        seed_rows.append({"method": method, **seed_result})
if seed_rows:
    display(pd.json_normalize(seed_rows))
else:
    print("Lượt chọn này chỉ đọc bộ M4/M5 đã lưu; xem báo cáo nguồn để truy vết khởi đầu.")


,method,status,selected_route_ids,source,history_path
0,M2,completed,"B04, C04, C16, C12, B20, C08, A10, A02, B02, C...",new_objective_greedy,work/data/processed/selection_benchmarks/48931...
1,M3,completed,"B16, C04, C08, C09, C12, C06, C05, C07, C15, C...",new_objective_greedy,work/data/processed/selection_benchmarks/48931...
2,M4,completed,"B04, C04, A10, B05, C12, C16, B16, B19, C08, C...",verified_saved_selection_history,work/data/processed/selection_benchmarks/48931...
3,M5,completed,"B04, C04, A10, B17, C12, C16, B16, B19, C08, C...",verified_saved_selection_history,work/data/processed/selection_benchmarks/48931...


,method,status,selected_indices,start,seed,selected_route_ids,metrics.D_combined,metrics.D_by_domain.inside,metrics.D_by_domain.outside,metrics.D_domain_mean,...,metrics.traversed_length_m,metrics.rc_traversed_length_m,metrics.represented_primary_strata,metrics.material_cells,metrics.touched_rc_cells,metrics.unique_rc_length_m,metrics.unique_rc_coverage_fraction,metrics.quantile_groups_covered,metrics.max_traversed_overlap_fraction,metrics.max_unique_corridor_overlap_fraction
0,M2,completed,"[23, 43, 55, 51, 39, 47, 9, 1, 21, 59, 27, 52,...",1,"[B04, C04]","[B04, C04, C16, C12, B20, C08, A10, A02, B02, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
1,M2,completed,"[27, 43, 47, 55, 51, 39, 9, 1, 5, 52, 59, 35, ...",2,"[B08, C04]","[B08, C04, C08, C16, C12, B20, A10, A02, A06, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
2,M2,completed,"[39, 43, 47, 55, 23, 51, 9, 1, 21, 59, 27, 52,...",3,"[B20, C04]","[B20, C04, C08, C16, B04, C12, A10, A02, B02, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
3,M2,completed,"[27, 51, 47, 9, 43, 39, 55, 1, 5, 52, 59, 35, ...",4,"[B08, C12]","[B08, C12, C08, A10, C04, B20, C16, A02, A06, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
4,M2,completed,"[35, 43, 47, 55, 51, 39, 9, 1, 59, 23, 52, 5, ...",5,"[B16, C04]","[B16, C04, C08, C16, C12, B20, A10, A02, C20, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
5,M2,completed,"[39, 51, 47, 55, 23, 43, 9, 1, 21, 59, 27, 52,...",6,"[B20, C12]","[B20, C12, C08, C16, B04, C04, A10, A02, B02, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
6,M2,completed,"[23, 51, 55, 43, 39, 47, 9, 1, 21, 59, 27, 52,...",7,"[B04, C12]","[B04, C12, C16, C04, B20, C08, A10, A02, B02, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
7,M2,completed,"[23, 47, 43, 39, 51, 55, 9, 1, 21, 59, 27, 52,...",8,"[B04, C08]","[B04, C08, C04, B20, C12, C16, A10, A02, B02, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
8,M2,completed,"[35, 51, 47, 55, 43, 39, 9, 1, 59, 23, 52, 5, ...",9,"[B16, C12]","[B16, C12, C08, C16, C04, B20, A10, A02, C20, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699
9,M2,completed,"[26, 43, 55, 51, 39, 47, 27, 9, 1, 52, 59, 23,...",10,"[B07, C04]","[B07, C04, C16, C12, B20, C08, B08, A10, A02, ...",0.377814,0.532860,0.310093,0.421477,...,266927.830347,257107.025047,"[A, B, C]",10,11,236092.404157,0.010528,33,0.510612,0.596699


### 6. Xem trạng thái và nơi lưu kết quả

Các CSV/JSON và manifest được ghi riêng; không chép kết quả đối chứng đè lên bộ 30 gốc. `benchmark_status` chỉ đọc báo cáo mới nhất. Nếu muốn chạy lại từ đầu kernel sau khi dừng, chạy cell thiết lập, cấu hình rồi cell chạy ở mục 3. Bộ nhớ đệm lưu theo phương pháp đã hoàn thành, không cam kết tiếp tục đúng giữa một lượt chọn.


In [14]:
for key, value in report["paths"].items():
    print(key, ":", value)
print(json.dumps(bench.benchmark_status(ROOT), ensure_ascii=False, indent=2))


comparison : work/data/processed/selection_benchmarks/489317219bd2c59527ba/comparison.csv
method_results : work/data/processed/selection_benchmarks/489317219bd2c59527ba/method_results.json
report : work/data/processed/selection_benchmarks/489317219bd2c59527ba/REPORT.md
manifest : work/data/processed/selection_benchmarks/489317219bd2c59527ba/manifest.json
directory : work/data/processed/selection_benchmarks/489317219bd2c59527ba
{
  "status": "completed_pre_survey_benchmarks",
  "fingerprint": "489317219bd2c59527ba36722e64d29933f22c9519e91cae5f189107ad516bac",
  "method_version": "fixed_pool_objective_benchmarks_v1",
  "started_at": "2026-11-05T02:33:41.137671+00:00",
  "finished_at": "2026-11-05T02:33:56.511062+00:00",
  "elapsed_seconds": 15.373397917021066,
  "requested_methods": [
    "M2",
    "M3",
    "M4",
    "M5"
  ],
  "methods": {
    "M2": {
      "status": "completed",
      "selected_indices": [
        23,
        43,
        55,
        51,
        39,
        47,
      

### Giới hạn diễn giải

M2/M3 dùng dữ liệu và gate tiền khảo sát đã lưu; M4/M5 được trích từ lịch sử cùng phiên bản. A×RC1 vẫn nằm trong mạng tham chiếu dù pool không tạo được tuyến đại diện. Không suy ra điều kiện thực địa, tính pháp lý của đa giác ranh giới hoặc chất lượng chu trình lái từ bảng này.

M0 ở [notebook 10](10_random_baseline.ipynb); biểu đồ M0 ở [notebook 11](11_random_comparison_plots.ipynb). M1 chưa được triển khai vì quy tắc chọn theo tỷ trọng chưa chốt.
